# 04 — Deep Dive: Transformers, Embeddings, Chunking & Distillation

Everything under the hood of the RAG pipeline, at working-engineer depth. Five parts:

- **Part A — The Transformer, end to end**: tokenization → embeddings → positional
  information → self-attention (with a from-scratch NumPy implementation) → multi-head
  attention → the full block → architecture families (encoder / decoder / enc-dec).
- **Part B — Embedding models in depth**: static → contextual → sentence embeddings,
  pooling, training objectives (contrastive / InfoNCE / triplets, hard negatives),
  bi- vs cross-encoders, similarity metrics, Matryoshka & instruction-tuned embeddings.
- **Part C — Embedding techniques beyond one dense vector**: sparse (TF-IDF / BM25 /
  SPLADE), multi-vector (ColBERT), hybrid retrieval + RRF (with a runnable demo),
  quantization and ANN indexes.
- **Part D — Chunking techniques**: fixed, recursive, token-based, structure-aware,
  semantic (implemented from scratch), and the advanced patterns (parent-document,
  contextual retrieval, late chunking).
- **Part E — Knowledge distillation**: soft targets & temperature, feature/relation
  distillation, DistilBERT / MiniLM (the model this pipeline uses **is** a distillation
  product), distillation for embeddings, plus a runnable dimensionality-reduction demo.

Every runnable cell uses only what's already installed for this project.

---
# Part A — The Transformer, end to end

## A.1 Why the Transformer exists

Before 2017, sequence models were RNNs/LSTMs (optionally with attention bolted on).
Two structural problems:

1. **Sequential computation** — token *t* cannot be processed before token *t−1*, so
   training cannot parallelize along the sequence dimension. GPUs sit idle.
2. **Long-range information decay** — information must survive being squeezed through a
   recurrent state, step by step. Even LSTMs struggle past a few hundred tokens.

*Attention Is All You Need* (Vaswani et al., 2017) removed recurrence entirely: every
token attends **directly** to every other token in one matrix multiplication. The path
length between any two positions becomes O(1) instead of O(n), and the whole sequence is
processed in parallel. That single change is why models scaled from millions to
trillions of parameters.

## A.2 Tokenization — the real input

A transformer never sees words. Text is split into **subword tokens** from a fixed
vocabulary (30k–250k entries), learned by one of three algorithms:

| Algorithm | Used by | Idea |
|---|---|---|
| **BPE** (byte-pair encoding) | GPT family, Llama | Iteratively merge the most frequent adjacent symbol pair |
| **WordPiece** | BERT, MiniLM | Like BPE but merges the pair that maximizes corpus likelihood |
| **Unigram / SentencePiece** | T5, many multilingual models | Start with a large vocab, prune tokens that hurt likelihood least |

Subwords are the compromise between characters (tiny vocab, very long sequences) and
words (huge vocab, out-of-vocabulary failures): frequent words stay whole, rare words
decompose (`"tokenization"` → `token ##ization`), so **nothing is ever out of
vocabulary**.

Engineering consequences you actually hit:
- Context limits, costs, and chunk sizes are all measured in **tokens**, not characters
  (English ≈ 4 chars/token; code and non-Latin scripts are worse).
- A model's tokenizer is part of the model. Mixing tokenizers between models = garbage.

In [1]:
# The actual tokenizer of the embedding model used by this pipeline (WordPiece).
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

text = "Retrieval augmented generation reduces hallucination."
encoding = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

print("Text  :", text)
print("Tokens:", tokens)
print("IDs   :", encoding["input_ids"])
print()
# Rare word -> subword decomposition; '##' marks a continuation piece.
print("'hallucination' alone:", tokenizer.tokenize("hallucination"))
print("Vocabulary size:", tokenizer.vocab_size)

Text  : Retrieval augmented generation reduces hallucination.
Tokens: ['[CLS]', 'retrieval', 'augmented', 'generation', 'reduces', 'hall', '##uc', '##ination', '.', '[SEP]']
IDs   : [101, 26384, 19335, 4245, 13416, 2534, 14194, 12758, 1012, 102]

'hallucination' alone: ['hall', '##uc', '##ination']
Vocabulary size: 30522


## A.3 Input embeddings and positional information

Each token ID indexes a row of a learned **embedding matrix** $E \in \mathbb{R}^{V \times d}$
(vocab size × model dimension; MiniLM: 30522 × 384). These rows are trained like any
other weight.

Self-attention is **permutation-invariant** — shuffle the tokens and (without position
info) the outputs shuffle identically. Order must therefore be injected explicitly:

| Scheme | Mechanism | Used by |
|---|---|---|
| **Sinusoidal** (original paper) | Add fixed $\sin/\cos$ waves of geometrically spaced frequencies to each embedding | original Transformer |
| **Learned absolute** | A trainable embedding per position index | BERT, MiniLM, GPT-2 |
| **RoPE** (rotary) | *Rotate* each Q/K pair by a position-dependent angle inside attention — relative offsets fall out of the dot product | Llama, Qwen, most modern LLMs |
| **ALiBi** | Add a distance-proportional penalty to attention scores | MPT, some long-context models |

Sinusoidal positional encoding, dimension pair $(2i, 2i+1)$ at position $pos$:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \qquad
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

The geometric frequency ladder lets the model express *relative* offsets as linear
functions of the encodings — and RoPE is what made very long context windows practical,
because rotations extrapolate/interpolate more gracefully than learned absolute slots.

In [2]:
import numpy as np

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> np.ndarray:
    """The original paper's fixed positional encoding matrix (max_len x d_model)."""
    positions = np.arange(max_len)[:, None]                # (max_len, 1)
    dims = np.arange(0, d_model, 2)[None, :]               # (1, d_model/2)
    angles = positions / np.power(10000, dims / d_model)   # broadcast
    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

pe = sinusoidal_positional_encoding(max_len=8, d_model=8)
np.set_printoptions(precision=3, suppress=True)
print("Positional encodings (8 positions x 8 dims):")
print(pe)
print()
# Early dims oscillate fast (fine-grained position), late dims slowly (coarse position).
print("dim 0 across positions (fast):", pe[:, 0])
print("dim 6 across positions (slow):", pe[:, 6])

Positional encodings (8 positions x 8 dims):
[[ 0.     1.     0.     1.     0.     1.     0.     1.   ]
 [ 0.841  0.54   0.1    0.995  0.01   1.     0.001  1.   ]
 [ 0.909 -0.416  0.199  0.98   0.02   1.     0.002  1.   ]
 [ 0.141 -0.99   0.296  0.955  0.03   1.     0.003  1.   ]
 [-0.757 -0.654  0.389  0.921  0.04   0.999  0.004  1.   ]
 [-0.959  0.284  0.479  0.878  0.05   0.999  0.005  1.   ]
 [-0.279  0.96   0.565  0.825  0.06   0.998  0.006  1.   ]
 [ 0.657  0.754  0.644  0.765  0.07   0.998  0.007  1.   ]]

dim 0 across positions (fast): [ 0.     0.841  0.909  0.141 -0.757 -0.959 -0.279  0.657]
dim 6 across positions (slow): [0.    0.001 0.002 0.003 0.004 0.005 0.006 0.007]


## A.4 Scaled dot-product self-attention — the core mechanism

Every token emits three projections of its embedding $x$:

- **Query** $q = xW_Q$ — *"what am I looking for?"*
- **Key** $k = xW_K$ — *"what do I contain / how can I be found?"*
- **Value** $v = xW_V$ — *"what do I contribute if selected?"*

For the whole sequence stacked into matrices:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Reading the formula:
1. $QK^\top$ — every query dotted with every key → an $n \times n$ **score matrix**
   (this is the $O(n^2)$ that makes long context expensive).
2. $/\sqrt{d_k}$ — dot products of high-dimensional vectors have variance $\propto d_k$;
   unscaled, softmax saturates into near one-hot outputs and gradients vanish. Dividing
   by $\sqrt{d_k}$ keeps scores in softmax's sensitive range.
3. $\text{softmax}$ row-wise → each row becomes a **probability distribution**: how much
   this token attends to every token.
4. $\cdot V$ — each output is a **weighted average of value vectors**. A token's new
   representation is literally a mixture of the tokens it chose to attend to.

**Causal masking** (decoder-only LLMs): add $-\infty$ to scores above the diagonal
before softmax, so position $t$ can only attend to positions $\le t$ — this is the only
difference that makes a transformer autoregressive.

In [3]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, causal=False):
    """The transformer's core equation, from scratch in NumPy."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)             # (n, n) similarity of queries to keys
    if causal:                                  # decoder-only: no attending to the future
        n = scores.shape[0]
        scores = np.where(np.triu(np.ones((n, n)), k=1).astype(bool), -1e9, scores)
    weights = softmax(scores, axis=-1)          # each row sums to 1
    return weights @ V, weights                 # outputs are weighted sums of values

# Tiny example: 4 tokens, d_model = 8. Token 3 ("it") is engineered to be similar
# to token 0 ("cat") so we can watch attention resolve the reference.
rng = np.random.default_rng(42)
d = 8
x_cat, x_sat, x_the = rng.normal(size=d), rng.normal(size=d), rng.normal(size=d)
x_it = x_cat + 0.35 * rng.normal(size=d)        # "it" nearly points at "cat"
X = np.stack([x_cat, x_sat, x_the, x_it])       # sequence: cat sat the it

# (Real models learn W_Q, W_K, W_V; identity here isolates the mechanism itself.)
out, weights = scaled_dot_product_attention(X, X, X)

tokens = ["cat", "sat", "the", "it"]
print("Attention weights (row = query token, column = attended token):")
print("        " + "".join(f"{t:>7}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>7} " + "".join(f"{w:7.3f}" for w in row))
print()
print(f"'it' puts {weights[3, 0]:.0%} of its attention on 'cat'"
      " -> its output vector becomes mostly cat's value vector.")

Attention weights (row = query token, column = attended token):
            cat    sat    the     it
    cat   0.445  0.035  0.074  0.446
    sat   0.146  0.473  0.172  0.209
    the   0.235  0.129  0.298  0.338
     it   0.379  0.042  0.091  0.488

'it' puts 38% of its attention on 'cat' -> its output vector becomes mostly cat's value vector.


In [4]:
# Same scores under a causal mask: no token can see to its right.
_, causal_weights = scaled_dot_product_attention(X, X, X, causal=True)
print("Causal attention weights (upper triangle forced to 0):")
print("        " + "".join(f"{t:>7}" for t in tokens))
for t, row in zip(tokens, causal_weights):
    print(f"{t:>7} " + "".join(f"{w:7.3f}" for w in row))

Causal attention weights (upper triangle forced to 0):
            cat    sat    the     it
    cat   1.000  0.000  0.000  0.000
    sat   0.236  0.764  0.000  0.000
    the   0.355  0.194  0.450  0.000
     it   0.379  0.042  0.091  0.488


## A.5 Multi-head attention

One attention head computes **one** mixing pattern per token. Language needs several at
once — syntax, coreference, positional locality, semantics. So the model runs $h$ heads
in parallel, each in a lower-dimensional subspace ($d_{head} = d_{model}/h$; MiniLM:
12 heads × 32 dims = 384):

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W_O,
\qquad \text{head}_i = \text{Attention}(QW_Q^i,\; KW_K^i,\; VW_V^i)$$

Each head gets its own learned projections, attends independently, and the concatenated
result is mixed by an output projection $W_O$. Interpretability work consistently finds
heads specializing: previous-token heads, syntactic-dependency heads, **induction heads**
(the pattern-copying circuit behind in-context learning).

**Inference-time evolution** (what you'll meet in production LLMs): the K and V tensors
of past tokens are cached (**KV cache** — the main VRAM consumer at long context);
**MQA/GQA** share K/V across head groups to shrink that cache; **FlashAttention**
reorders the computation to avoid materializing the $n \times n$ matrix in slow memory.

In [5]:
def multi_head_attention(X, num_heads, rng):
    """Multi-head attention from scratch (random projections for demonstration)."""
    n, d_model = X.shape
    d_head = d_model // num_heads
    heads = []
    for _ in range(num_heads):
        # Per-head learned projections into a d_head-dimensional subspace
        W_q, W_k, W_v = (rng.normal(size=(d_model, d_head)) / np.sqrt(d_model)
                         for _ in range(3))
        head_out, _ = scaled_dot_product_attention(X @ W_q, X @ W_k, X @ W_v)
        heads.append(head_out)
    W_o = rng.normal(size=(d_model, d_model)) / np.sqrt(d_model)
    return np.concatenate(heads, axis=-1) @ W_o    # concat -> output projection

out = multi_head_attention(X, num_heads=2, rng=np.random.default_rng(0))
print("Input shape :", X.shape, " -> output shape:", out.shape)
print()
print("MiniLM-L6-v2 for comparison: 6 layers x 12 heads x 32 dims/head = 384 d_model")

Input shape : (4, 8)  -> output shape: (4, 8)

MiniLM-L6-v2 for comparison: 6 layers x 12 heads x 32 dims/head = 384 d_model


## A.6 The rest of the transformer block

Attention alone is just mixing. Each block wraps it with three more ingredients:

```
x ──▶ LayerNorm ──▶ Multi-Head Attention ──▶ (+x residual) ──▶
      LayerNorm ──▶ Feed-Forward Network ──▶ (+ residual) ──▶ next block
```

- **Feed-forward network (FFN)**: two linear layers with a nonlinearity,
  $\text{FFN}(x) = \text{GELU}(xW_1)W_2$, expanding to ~4× $d_{model}$ and back.
  Applied **per token** independently. Roughly ⅔ of a transformer's parameters live
  here, and mechanistic work treats FFN layers as the model's **key–value memory** —
  where factual associations are stored (attention *routes*, FFN *transforms/recalls*).
- **Residual connections** ($x + \text{sublayer}(x)$): gradients flow through the
  identity path, making 100+ layer stacks trainable. Each block *edits* the running
  representation ("residual stream") rather than replacing it.
- **LayerNorm**: stabilizes activation scale. Modern models use **pre-LN** (norm before
  the sublayer — trains stably without warmup tricks) vs. the original **post-LN**.
  Llama-family models use RMSNorm, a cheaper variant.

## A.7 The three architecture families

| Family | Attention | Pretraining objective | Best at | Examples |
|---|---|---|---|---|
| **Encoder-only** | bidirectional | Masked LM (cloze) | understanding/representation: embeddings, classification, NER | BERT, RoBERTa, **MiniLM** |
| **Decoder-only** | causal | next-token prediction | generation | GPT-x, Llama, Claude, Groq-hosted models |
| **Encoder-decoder** | bidirectional enc + causal dec + cross-attention | denoising / span corruption | seq-to-seq: translation, summarization | T5, BART |

This maps exactly onto our RAG pipeline: the **embedding model is encoder-only**
(every token sees full bidirectional context — ideal for representing meaning), and the
**generator LLM is decoder-only** (built to produce text). RAG glues the two together
through the vector store.

---
# Part B — Embedding models in depth

## B.1 From static to contextual embeddings

**Static era (2013–2017):** word2vec (skip-gram / CBOW trained with negative sampling),
GloVe (factorizing corpus co-occurrence statistics), FastText (character n-grams, so
misspellings/rare words still get vectors). One fixed vector **per word type** — which
means "bank" (river) and "bank" (finance) share a single point in space. Famous linear
structure: $\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$.

**Contextual era (2018–):** ELMo, then BERT — the vector for each token is computed
**from its sentence**, so the two "bank"s land in different places. Every modern
embedding model is a transformer encoder producing contextual token vectors.

## B.2 From token vectors to sentence vectors — pooling and SBERT

A retrieval system needs **one vector per text**, not per token. Two findings shaped
everything:

1. Naively averaging raw BERT token vectors (or taking its `[CLS]` token) produces
   **worse** sentence similarity than 2014-era GloVe averages. Raw MLM training never
   optimizes for a meaningful sentence geometry, and the space is *anisotropic*
   (vectors crowd a narrow cone, inflating all cosine similarities).
2. **Sentence-BERT** (Reimers & Gurevych, 2019) fixed it: fine-tune the encoder in a
   **siamese (bi-encoder)** setup — the same network embeds two texts, a similarity
   objective pulls related pairs together — then **mean-pool** token vectors into one.
   This is exactly the `sentence-transformers` library this pipeline uses.

Pooling options: **mean pooling** (default, robust — used by MiniLM), `[CLS]` pooling
(some models train the CLS token specifically for this), max pooling (rare). The cell
below reproduces MiniLM's mean pooling by hand and verifies it matches the library.

In [6]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModel

sentence = "Vector databases enable semantic search."

# --- by hand: raw transformer -> token vectors -> masked mean pooling -> normalize
tok = tokenizer(sentence, return_tensors="pt")
encoder = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
with torch.no_grad():
    token_vectors = encoder(**tok).last_hidden_state        # (1, n_tokens, 384)

mask = tok["attention_mask"].unsqueeze(-1).float()           # ignore padding positions
mean_pooled = (token_vectors * mask).sum(1) / mask.sum(1)    # (1, 384)
manual = torch.nn.functional.normalize(mean_pooled, dim=1)[0].numpy()

# --- the library doing the same thing
st_model = SentenceTransformer("all-MiniLM-L6-v2")
library = st_model.encode([sentence], normalize_embeddings=True)[0]

print("token vectors shape:", tuple(token_vectors.shape), "-> pooled:", manual.shape)
print("max |manual - library| =", float(np.abs(manual - library).max()))
print("=> a sentence embedding is just masked mean pooling over contextual token vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

token vectors shape: (1, 8, 384) -> pooled: (384,)
max |manual - library| = 1.3785393093712628e-07
=> a sentence embedding is just masked mean pooling over contextual token vectors


## B.3 How embedding models are trained — contrastive learning

The training goal is geometric: **related texts close, unrelated texts far**. The loss
family is contrastive:

- **Triplet loss**: for (anchor $a$, positive $p$, negative $n$), enforce
  $\;d(a,p) + \text{margin} < d(a,n)$.
- **InfoNCE / Multiple-Negatives Ranking Loss (MNRL)** — the workhorse. A batch of
  (query, positive-passage) pairs; every *other* passage in the batch is a free
  negative ("in-batch negatives"):

$$\mathcal{L} = -\log \frac{e^{\,\text{sim}(q, p^+)/\tau}}{\sum_{j} e^{\,\text{sim}(q, p_j)/\tau}}$$

  — i.e. softmax-classify the true passage among all passages in the batch. The
  **temperature** $\tau$ sharpens the distribution; large batches matter because more
  in-batch negatives = a harder, more informative task (why embedding models train on
  TPU pods with batch sizes in the thousands).

- **Hard negative mining** (met in notebook 02's sample data): random negatives quickly
  become trivial. Mine passages that are *lexically similar but wrong* — from BM25, or
  from an earlier checkpoint of the model itself — and add them explicitly to the
  denominator. This forces fine-grained distinctions and is the single biggest quality
  lever in modern recipes (E5, GTE, BGE, NV-Embed all lean on it).

Typical modern recipe: (1) weakly-supervised contrastive pretraining on billions of
mined pairs (title/body, question/answer, ...) → (2) fine-tuning on curated datasets
(MS MARCO, NLI, ...) with hard negatives, often with **cross-encoder distillation**
(Part E).

## B.4 Bi-encoder vs cross-encoder — the retrieval/rerank split

| | Bi-encoder | Cross-encoder |
|---|---|---|
| Input | query and doc **separately** | query and doc **concatenated** |
| Output | one vector each → cosine | one relevance score |
| Token interaction | none across texts | full attention across both |
| Quality | good | best |
| Cost at query time | O(1) encode + ANN lookup | O(candidates) full forward passes |
| Precompute? | **yes** — the whole point of a vector DB | no |

Production pattern: **retrieve** top-100 with a bi-encoder (this pipeline), then
**rerank** the top-100 with a cross-encoder (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`
or a hosted reranker) and keep the top-5 for the LLM. Reranking is usually the
cheapest large quality win available to a RAG system.

## B.5 Similarity metrics — and a subtlety that bites

- **Cosine similarity**: angle only; scale-invariant. The default for text.
- **Dot product**: cosine × magnitudes — some models (e.g. DPR) train with it and encode
  "importance" in vector length.
- **Euclidean (L2)**: what our FAISS `IndexFlatL2` computes.

For **unit-normalized** vectors the three agree on ranking:
$\|a-b\|^2 = 2 - 2\cos(a,b)$ — L2 ranking is exactly reversed cosine ranking.
The subtlety: `SentenceTransformer.encode` does **not** normalize by default. MiniLM's
outputs are near unit length so L2 works well here, but the safe production habit is
`normalize_embeddings=True` + inner-product index. Always match the metric to what the
model was trained with (its model card says).

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

pairs = [
    ("How do I reset my password?", "Steps to recover your account credentials"),
    ("How do I reset my password?", "The attention mechanism maps queries to key-value pairs"),
    ("The cat sat on the mat.",     "A feline was resting on the rug."),
]
for a, b in pairs:
    va, vb = st_model.encode([a, b])
    print(f"{cosine_similarity([va], [vb])[0, 0]:.3f}  |  {a!r:45} vs {b!r}")
print()
norms = np.linalg.norm(st_model.encode([p[0] for p in pairs]), axis=1)
print("Vector norms (unnormalized encode):", norms.round(3),
      "-> near 1.0, but don't rely on it; pass normalize_embeddings=True")

0.651  |  'How do I reset my password?'                 vs 'Steps to recover your account credentials'
-0.007  |  'How do I reset my password?'                 vs 'The attention mechanism maps queries to key-value pairs'
0.563  |  'The cat sat on the mat.'                     vs 'A feline was resting on the rug.'

Vector norms (unnormalized encode): [1. 1. 1.] -> near 1.0, but don't rely on it; pass normalize_embeddings=True


## B.6 Modern embedding-model topics worth knowing

- **Matryoshka Representation Learning (MRL)**: train so every *prefix* of the vector
  (first 64, 128, 256... dims) is itself a valid embedding. Ship 1024-dim quality,
  store/search 256-dim cost, rerank with full dims. (OpenAI `text-embedding-3` exposes
  this as the `dimensions` parameter.) Part E has a poor-man's version of this idea.
- **Instruction-tuned / prefixed embeddings**: E5/BGE-family models expect
  `"query: ..."` vs `"passage: ..."` prefixes (asymmetric search); newer models take
  free-form task instructions. **Forgetting the prefix silently costs ~5-10 points of
  recall** — read the model card.
- **Long-context embeddings**: MiniLM truncates at 256 word pieces (silently!). Models
  like `jina-embeddings` or `nomic-embed-text` handle 8k+. Know your model's limit —
  it's a chunking constraint (Part D).
- **Benchmarks**: **MTEB** is the standard leaderboard (retrieval, reranking,
  clustering, STS...). Evaluate on *your* data anyway: a small model tuned to your
  domain often beats a leaderboard giant.
- **Choosing in practice**: MiniLM-class (384d) for latency-critical/prototyping;
  mpnet/BGE-base-class (768d) as the quality default; large/API models when recall is
  the product. Embedding dimension × corpus size × 4 bytes = your index RAM bill.

---
# Part C — Embedding techniques beyond a single dense vector

## C.1 Sparse representations — TF-IDF, BM25, SPLADE

Before (and alongside) dense vectors: represent text as a **vocabulary-sized sparse
vector** of term weights.

- **TF-IDF**: weight = term frequency × inverse document frequency (rare terms matter
  more).
- **BM25** — the battle-tested default of search engines. Per query term $t$ in doc $d$:

$$\text{score}(t,d) = \text{IDF}(t) \cdot
\frac{tf_{t,d}\,(k_1+1)}{tf_{t,d} + k_1\,(1 - b + b\cdot\frac{|d|}{\text{avgdl}})}$$

  i.e. TF-IDF with **term-frequency saturation** ($k_1$: the 10th occurrence adds little)
  and **length normalization** ($b$: long docs don't win by volume).
- **SPLADE**: a learned sparse model — a transformer predicts weights over the *whole
  vocabulary* including terms **not present** in the text (learned expansion), keeping
  inverted-index speed with neural quality.

**The core trade-off**: sparse = exact term matching — unbeatable for IDs, error codes,
part numbers, names ("TRCY-6312" will never be found semantically); dense = meaning —
finds "account recovery" for "password reset". Real systems want both →

## C.2 Hybrid retrieval + Reciprocal Rank Fusion

Run sparse and dense in parallel and fuse rankings. **RRF** is the standard fusion —
score-scale-free, no tuning:

$$\text{RRF}(d) = \sum_{r \,\in\, \text{rankers}} \frac{1}{k + \text{rank}_r(d)}, \qquad k = 60$$

In [8]:
# Hybrid retrieval demo on this project's own knowledge base.
import sys
sys.path.append("..")
from sklearn.feature_extraction.text import TfidfVectorizer
from src.data_loader import load_all_documents

docs = load_all_documents("../data")
corpus = [d.page_content for d in docs]
labels = [f"{d.metadata['source_file']}" for d in docs]

# Sparse ranker: TF-IDF (BM25's simpler cousin; same exact-term behaviour)
tfidf = TfidfVectorizer(stop_words="english")
doc_tfidf = tfidf.fit_transform(corpus)

# Dense ranker: our MiniLM embeddings
doc_dense = st_model.encode(corpus, normalize_embeddings=True, show_progress_bar=False)

def ranks(scores):
    order = np.argsort(-scores)
    r = np.empty_like(order); r[order] = np.arange(1, len(order) + 1)
    return r

def hybrid_search(query, k=60, top=3):
    sparse_scores = (tfidf.transform([query]) @ doc_tfidf.T).toarray()[0]
    dense_scores = doc_dense @ st_model.encode([query], normalize_embeddings=True)[0]
    rrf = 1 / (k + ranks(sparse_scores)) + 1 / (k + ranks(dense_scores))
    for i in np.argsort(-rrf)[:top]:
        print(f"  sparse#{ranks(sparse_scores)[i]:>2} dense#{ranks(dense_scores)[i]:>2} "
              f"rrf={rrf[i]:.4f}  {labels[i]}: {corpus[i][:70].replace(chr(10), ' ')}")

print("Query: 'model with 131072 context window'   (exact number -> sparse wins)")
hybrid_search("model with 131072 context window")
print()
print("Query: 'how to stop the model from making things up'   (pure semantics -> dense wins)")
hybrid_search("how to stop the model from making things up")

Found 3 PDF files
  loaded attention_is_all_you_need.pdf: 3 pages
  loaded embedding_models_report.pdf: 3 pages
  loaded rag_systems_overview.pdf: 2 pages
Found 2 text files
  loaded machine_learning.txt
  loaded python_intro.txt
Found 1 CSV files
  loaded ai_models.csv: 4 rows
Found 1 JSON files
  loaded rag_faq.json: 4 entries
Found 1 Excel files
  loaded vector_databases.xlsx: 4 rows
Found 1 SQLite databases
  loaded llm_catalog.db: 4 rows from 1 table(s)
Total documents loaded: 26


Query: 'model with 131072 context window'   (exact number -> sparse wins)
  sparse# 1 dense# 1 rrf=0.0328  llm_catalog.db: name: claude-sonnet provider: Anthropic context_window: 200000 open_we
  sparse# 2 dense# 7 rrf=0.0311  llm_catalog.db: name: llama-3.1-8b-instant provider: Groq context_window: 131072 open_
  sparse# 8 dense# 3 rrf=0.0306  llm_catalog.db: name: gpt-4o provider: OpenAI context_window: 128000 open_weights: no 

Query: 'how to stop the model from making things up'   (pure semantics -> dense wins)
  sparse# 2 dense# 1 rrf=0.0325  ai_models.csv: model_name: all-MiniLM-L6-v2 model_type: embedding dimensions: 384 pro
  sparse# 5 dense# 6 rrf=0.0305  embedding_models_report.pdf: Hard Negative Mining Techniques Hard negative mining is a technique fo
  sparse#10 dense# 2 rrf=0.0304  rag_faq.json: Q: How does RAG reduce hallucination? A: The prompt instructs the LLM 


## C.3 Multi-vector: ColBERT late interaction

A middle path between bi- and cross-encoders. Store **one vector per token** of each
document; at query time compute *MaxSim*:
$\text{score}(q,d) = \sum_{i \in q} \max_{j \in d} \; q_i \cdot d_j$ — each query token
finds its best-matching document token. Near-cross-encoder quality, still indexable —
at the cost of ~2 orders of magnitude more storage (mitigated by PLAID/ColBERTv2
compression). Worth knowing as the quality ceiling for retrieval-heavy products.

## C.4 Making dense search scale — ANN indexes and compression

Our `IndexFlatL2` scans every vector — **exact** and fine to ~1M vectors. Beyond that:

| Technique | Idea | Trade-off |
|---|---|---|
| **HNSW** | multi-layer navigable small-world graph | the default ANN; fast + accurate, RAM-hungry (Chroma uses it) |
| **IVF** | cluster vectors; probe only `nprobe` nearest clusters | tunable speed/recall, needs training pass |
| **PQ** (product quantization) | split vector into subvectors, quantize each to a codebook | 10-100× compression, approximate distances |
| **Scalar/binary quantization** | float32 → int8 (4×) or 1 bit/dim (32×) | int8 nearly free; binary needs rescoring pass |

Standard recipe at scale: HNSW or IVF+PQ for the candidate sweep, exact re-scoring of
the top candidates with full-precision vectors — plus MRL (B.6) to shrink dims first.

---
# Part D — Chunking techniques

## D.1 Why chunking is a *retrieval* decision, not a storage one

Three forces, in tension:

1. **Model limits** — the embedding model truncates (MiniLM: 256 word pieces ≈ 800-1000
   chars — chunks beyond that are **silently cut**, a classic invisible bug).
2. **Embedding dilution** — one vector must summarize the whole chunk. A chunk mixing
   five topics embeds as a mush that matches none of them well. Small chunks = sharp
   vectors = **precision**.
3. **Context sufficiency** — the LLM needs enough surrounding material to answer.
   Large chunks = **recall/completeness**.

So: **small chunks retrieve better, large chunks answer better** — every strategy below
is a different resolution of that conflict. (Our pipeline's 1000/200 recursive split is
the standard starting point, not a law.)

In [9]:
# A composite document that deliberately switches topics — our chunking test bench.
composite_text = """The transformer architecture relies entirely on attention mechanisms. \
Each token attends to every other token through scaled dot-product attention. \
Multi-head attention lets the transformer capture several relation types in parallel.

Cooking pasta properly starts with generously salted boiling water. \
The pasta should finish cooking directly in the sauce for the best flavour. \
Fresh basil and olive oil complete a classic pasta sauce.

The French Revolution began in 1789 and transformed European politics. \
Revolutionary crowds stormed the Bastille prison in Paris. \
Within months the revolution had abolished feudal privileges across France."""

print(composite_text[:180], "...")
print()
print("Length:", len(composite_text), "characters, 3 distinct topics"
      " (transformers, cooking, history)")

The transformer architecture relies entirely on attention mechanisms. Each token attends to every other token through scaled dot-product attention. Multi-head attention lets the tr ...

Length: 643 characters, 3 distinct topics (transformers, cooking, history)


## D.2 Fixed-size vs recursive character splitting

- **Fixed-size** (`CharacterTextSplitter`): cut every N characters on a single
  separator. Simple, but happily bisects sentences and ideas.
- **Recursive** (`RecursiveCharacterTextSplitter` — what this pipeline uses): try
  separators in order `["\n\n", "\n", " ", ""]` — split on paragraphs if the pieces fit,
  else lines, else words. Chunks end at the most natural boundary available. **Overlap**
  (ours: 200) repeats the tail of each chunk so boundary-straddling facts exist intact
  in at least one chunk.

In [10]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

fixed = CharacterTextSplitter(separator=" ", chunk_size=220, chunk_overlap=0)
recursive = RecursiveCharacterTextSplitter(chunk_size=220, chunk_overlap=40,
                                           separators=["\n\n", "\n", " ", ""])

print("FIXED-SIZE (breaks mid-idea):")
for c in fixed.split_text(composite_text):
    print("  |", c[:95].replace("\n", " "), "...")
print()
print("RECURSIVE (respects paragraph boundaries where it can):")
for c in recursive.split_text(composite_text):
    print("  |", c[:95].replace("\n", " "), "...")

FIXED-SIZE (breaks mid-idea):
  | The transformer architecture relies entirely on attention mechanisms. Each token attends to eve ...
  | in parallel.  Cooking pasta properly starts with generously salted boiling water. The pasta sho ...
  | French Revolution began in 1789 and transformed European politics. Revolutionary crowds stormed ...

RECURSIVE (respects paragraph boundaries where it can):
  | The transformer architecture relies entirely on attention mechanisms. Each token attends to eve ...
  | capture several relation types in parallel. ...
  | Cooking pasta properly starts with generously salted boiling water. The pasta should finish coo ...
  | The French Revolution began in 1789 and transformed European politics. Revolutionary crowds sto ...


### D.2.1 The recursive algorithm, step by step

Worth understanding precisely, because it's the default in almost every RAG stack.
`RecursiveCharacterTextSplitter._split_text(text, separators)` does this:

1. **Choose a separator.** Walk the separator list *in order* and take the **first one
   that actually occurs** in this text. Remember the *remaining* (finer) separators —
   they are what recursion will use. If none occur, use the **last** separator in the
   list. An empty string `""` short-circuits immediately and means "split between
   characters".
2. **Split** the text on the chosen separator.
3. **Classify each piece:**
   - `len(piece) < chunk_size` → it's acceptable; push it into a buffer of "good splits".
   - `len(piece) >= chunk_size` → it's still too big: flush the buffer, then **recurse
     on that piece with the finer separators**. *If no finer separators remain, the
     oversized piece is emitted as-is.*
4. **Merge back up** (`_merge_splits`): greedily pack the buffered small pieces into
   chunks up to `chunk_size`. When the next piece would overflow, emit the chunk, then
   **pop whole pieces off the front** until the retained tail fits in `chunk_overlap` —
   that retained tail becomes the head of the next chunk. **That is how overlap is
   implemented.**

So the recursion goes *coarse → fine*: try to break at paragraphs; only the paragraphs
that are still too long get broken at newlines; only those still too long get broken at
spaces; and so on. The result is the **largest semantically-coherent unit that fits**.

Two consequences that surprise people, both demonstrated below:

- **`chunk_size` is a target, not a guarantee.** An indivisible run longer than
  `chunk_size` is emitted whole (with a logged warning) unless `""` is in your
  separator list. Base64 blobs, long URLs, and minified code hit this.
- **`chunk_overlap` is granular to the split units, not to characters.** Overlap can
  only retain *whole* pieces that fit the overlap budget — so word-level splits give
  near-exact overlap, while paragraph-level splits can give you **zero**.

In [11]:
# Consequence 1: chunk_size is a target, not a guarantee.
unsplittable = "A" * 120                       # contains none of the separators
no_char_fallback = RecursiveCharacterTextSplitter(
    chunk_size=50, chunk_overlap=0, separators=["\n\n", "\n", " "]      # no "" !
)
with_char_fallback = RecursiveCharacterTextSplitter(
    chunk_size=50, chunk_overlap=0, separators=["\n\n", "\n", " ", ""]  # "" present
)
print("separators without '':", [len(c) for c in no_char_fallback.split_text(unsplittable)],
      "<- exceeds chunk_size, emitted as-is")
print("separators with    '':", [len(c) for c in with_char_fallback.split_text(unsplittable)],
      "<- respected")

separators without '': [120] <- exceeds chunk_size, emitted as-is
separators with    '': [50, 50, 20] <- respected


In [12]:
# Consequence 2: overlap retains whole split units, so its size depends on them.
def measure_overlaps(chunks):
    """Longest suffix of each chunk that is a prefix of the next chunk."""
    out = []
    for a, b in zip(chunks, chunks[1:]):
        shared = next((a[-k:] for k in range(min(len(a), len(b)), 0, -1)
                       if b.startswith(a[-k:])), "")
        out.append(len(shared))
    return out

# Small units (words) -> overlap lands close to the requested 20 characters
word_text = " ".join(f"w{i:02d}" for i in range(40))
word_chunks = RecursiveCharacterTextSplitter(
    chunk_size=60, chunk_overlap=20, separators=[" "]
).split_text(word_text)

# Large units (whole 48-char paragraphs) -> no whole unit fits a 20-char budget
para_text = "\n\n".join(f"P{i} " + "x" * 45 for i in range(4))
para_chunks = RecursiveCharacterTextSplitter(
    chunk_size=60, chunk_overlap=20, separators=["\n\n", " ", ""]
).split_text(para_text)

print("word-level splits,      requested overlap 20 -> actual:", measure_overlaps(word_chunks))
print("paragraph-level splits, requested overlap 20 -> actual:", measure_overlaps(para_chunks))
print()
print("Overlap is only ever whole split units that fit the budget -- never a hard promise.")

word-level splits,      requested overlap 20 -> actual: [19, 19, 19]
paragraph-level splits, requested overlap 20 -> actual: [0, 0, 0]

Overlap is only ever whole split units that fit the budget -- never a hard promise.


### D.2.2 Separator strategies per content type

The separator list *is* the strategy: it encodes "what counts as a natural boundary in
this kind of text", ordered from most to least preferred. Tailoring it to your content
is the cheapest chunking win available.

| Content | Separators (coarse → fine) | Why |
|---|---|---|
| **Prose / default** | `["\n\n", "\n", " ", ""]` | paragraph → line → word → character (this pipeline) |
| **Markdown** | `["\n## ", "\n### ", "\n\n", "\n", " ", ""]` | prefer section headers over blank lines |
| **Python** | `["\nclass ", "\ndef ", "\n\tdef ", "\n\n", "\n", " ", ""]` | keep classes/functions intact |
| **HTML** | `["<div", "<p", "<br", "\n", " ", ""]` | break on block elements |
| **CJK / no spaces** | `["\n\n", "\n", "。", "，", ""]` | space is not a word boundary; use CJK punctuation |
| **Tabular rows** | `["\n"]` (or one Document per row) | a row is the atomic unit — see notebook 03 |

LangChain ships the language presets:
`RecursiveCharacterTextSplitter.from_language(Language.PYTHON)` (also `MARKDOWN`, `JS`,
`HTML`, `LATEX`, ...). Inspect any of them with
`RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)`.

**Tuning `chunk_size` / `chunk_overlap`:**

- `chunk_size` — must fit the embedding model's real limit (MiniLM: 256 word pieces
  ≈ 800–1000 chars, so our 1000 sits right at the edge; measure in *tokens* via
  `from_huggingface_tokenizer` if content tokenizes badly). Smaller = sharper vectors
  and better precision; larger = more self-contained context for the LLM.
- `chunk_overlap` — the usual rule of thumb is **10–20% of `chunk_size`** (ours: 200 of
  1000 = 20%). It's insurance against a fact being severed at a boundary; overlap costs
  storage and creates near-duplicate retrieval hits, so don't exceed ~25%.
- Both are **retrieval hyperparameters** — treat them as things you *measure*, and
  remember that changing either requires **re-embedding the whole corpus**.

In [13]:
# LangChain's built-in language presets — look at what Python's separators encode.
from langchain_text_splitters import Language

print("PYTHON:  ", RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON))
print("MARKDOWN:", RecursiveCharacterTextSplitter.get_separators_for_language(Language.MARKDOWN)[:6], "...")
print()

code_sample = '''class EmbeddingPipeline:
    def chunk_documents(self, documents):
        """Split documents into chunks."""
        return self.splitter.split_documents(documents)

    def embed_chunks(self, chunks):
        """Embed each chunk."""
        return self.model.encode([c.page_content for c in chunks])
'''

py_splitter = RecursiveCharacterTextSplitter.from_language(
    Language.PYTHON, chunk_size=170, chunk_overlap=0
)
for i, chunk in enumerate(py_splitter.split_text(code_sample), 1):
    print(f"--- chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
print("Each method stays whole: '\\ndef ' outranks '\\n\\n', so the split lands at the")
print("method boundary. (Shrink chunk_size below a method's length and it breaks apart")
print("anyway -- the separator list expresses a preference, not a hard constraint.)")

PYTHON:   ['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']
MARKDOWN: ['\n#{1,6} ', '```\n', '\n\\*\\*\\*+\n', '\n---+\n', '\n___+\n', '\n\n'] ...

--- chunk 1 (165 chars) ---
class EmbeddingPipeline:
    def chunk_documents(self, documents):
        """Split documents into chunks."""
        return self.splitter.split_documents(documents)
--- chunk 2 (130 chars) ---
def embed_chunks(self, chunks):
        """Embed each chunk."""
        return self.model.encode([c.page_content for c in chunks])
Each method stays whole: '\ndef ' outranks '\n\n', so the split lands at the
method boundary. (Shrink chunk_size below a method's length and it breaks apart
anyway -- the separator list expresses a preference, not a hard constraint.)


## D.3 Token-based chunking

Character counts are a proxy; the model's truncation limit is in **tokens**. Splitting
by the *model's own tokenizer* guarantees chunks actually fit — essential when content
tokenizes unpredictably (code, URLs, non-English).

In [14]:
token_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,            # MiniLM's own WordPiece tokenizer from Part A
    chunk_size=64,        # measured in TOKENS now, not characters
    chunk_overlap=12,
)
token_chunks = token_splitter.split_text(composite_text)
for c in token_chunks:
    n_tokens = len(tokenizer.tokenize(c))
    print(f"  [{n_tokens:>2} tokens] {c[:80].replace(chr(10), ' ')}...")
print()
print("Every chunk is guaranteed under the model's real limit (MiniLM: 256 word pieces)")

  [39 tokens] The transformer architecture relies entirely on attention mechanisms. Each token...
  [37 tokens] Cooking pasta properly starts with generously salted boiling water. The pasta sh...
  [33 tokens] The French Revolution began in 1789 and transformed European politics. Revolutio...

Every chunk is guaranteed under the model's real limit (MiniLM: 256 word pieces)


## D.4 Structure-aware chunking

Documents with markup carry their own semantic boundaries — use them. Splitting
markdown by headers keeps each section whole **and** puts the section path into
metadata (which then rides along into the vector store, retrievable and filterable —
the metadata lesson from notebook 01 again).

In [15]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """# RAG Handbook
## Ingestion
Parse files into Documents, then chunk them.
### Loaders
PDF, CSV, JSON and SQL loaders all emit the same structure.
## Retrieval
Embed the query, search the vector store, assemble context."""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")]
)
for d in md_splitter.split_text(markdown_doc):
    print("metadata:", d.metadata)
    print("content :", d.page_content.replace("\n", " "))
    print()

metadata: {'h1': 'RAG Handbook', 'h2': 'Ingestion'}
content : Parse files into Documents, then chunk them.

metadata: {'h1': 'RAG Handbook', 'h2': 'Ingestion', 'h3': 'Loaders'}
content : PDF, CSV, JSON and SQL loaders all emit the same structure.

metadata: {'h1': 'RAG Handbook', 'h2': 'Retrieval'}
content : Embed the query, search the vector store, assemble context.



## D.5 Semantic chunking — split where the *meaning* shifts

Instead of counting characters, embed each sentence and cut where consecutive
sentences' similarity **drops** — a topic boundary. This is what LangChain's
experimental `SemanticChunker` does; the from-scratch version below shows the whole
mechanism in ~20 lines. Costs one embedding pass over every sentence at ingestion
time; shines on topic-hopping documents (meeting notes, wikis, transcripts).

In [16]:
import re

def semantic_chunk(text, model, percentile=25):
    """Split text where consecutive-sentence similarity drops below a percentile."""
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    vectors = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)

    # Similarity between each sentence and the next
    sims = np.array([float(vectors[i] @ vectors[i + 1]) for i in range(len(vectors) - 1)])
    threshold = np.percentile(sims, percentile)   # the weakest links are boundaries

    chunks, current = [], [sentences[0]]
    for i, sim in enumerate(sims):
        if sim < threshold:
            chunks.append(" ".join(current)); current = []
        current.append(sentences[i + 1])
    chunks.append(" ".join(current))
    return chunks, sims, threshold

chunks, sims, threshold = semantic_chunk(composite_text.replace("\n\n", " "), st_model)

print(f"Consecutive-sentence similarities (threshold {threshold:.3f}):")
print(" ", sims.round(3))
print()
for i, c in enumerate(chunks, 1):
    print(f"CHUNK {i}: {c[:90]}...")
print()
print("The two deepest similarity dips are exactly the topic changes"
      " (transformers -> cooking -> French Revolution).")

Consecutive-sentence similarities (threshold 0.231):
  [0.295 0.373 0.041 0.62  0.626 0.004 0.379 0.318]

CHUNK 1: The transformer architecture relies entirely on attention mechanisms. Each token attends t...
CHUNK 2: Cooking pasta properly starts with generously salted boiling water. The pasta should finis...
CHUNK 3: The French Revolution began in 1789 and transformed European politics. Revolutionary crowd...

The two deepest similarity dips are exactly the topic changes (transformers -> cooking -> French Revolution).


## D.6 Advanced patterns — decoupling what you *search* from what you *send*

The precision-vs-context conflict of D.1 dissolves once the **retrieval unit** and the
**generation unit** are allowed to differ:

| Pattern | Mechanism | When |
|---|---|---|
| **Parent-document / small-to-big** | embed small child chunks; on a hit, hand the LLM the larger parent chunk | the default upgrade for most RAG systems |
| **Sentence-window** | embed single sentences; return ±k surrounding sentences | precision-critical QA |
| **Contextual retrieval** (Anthropic) | before embedding, an LLM prepends 1-2 sentences situating the chunk in its document ("This chunk is from the Q3 filing, discussing...") | chunks that are ambiguous out of context; large recall gains, one-off LLM ingestion cost |
| **Late chunking** | run the long-context embedding model over the *whole* document, then pool token vectors per chunk — each chunk's vector has seen full-document context | needs a long-context embedding model |
| **Propositions** | LLM decomposes text into atomic self-contained facts; embed those | maximum-precision fact QA, expensive |
| **Hierarchical / RAPTOR** | cluster chunks, LLM-summarize clusters, embed summaries too; retrieve across levels | questions needing document-level synthesis |

Practical guidance: start 1000/200 recursive (this pipeline) → move to structure-aware
+ parent-document as content grows → add semantic/contextual only where measured recall
says so. **Chunking changes require re-embedding the corpus — always version your
ingestion configuration** alongside the index.

---
# Part E — Knowledge distillation

## E.1 The idea

Train a small **student** to imitate a large **teacher** — not the ground-truth labels,
the *teacher's behavior*. The student inherits most of the quality at a fraction of the
size/latency/cost. (Hinton, Vinyals & Dean 2015, "Distilling the Knowledge in a Neural
Network".)

Why imitating beats training from scratch: hard labels say "this is a cat: 1, everything
else: 0". The teacher's full output distribution says "cat 0.90, dog 0.07, car 0.0001" —
the **relative probabilities of the wrong answers** ("dark knowledge") encode the
similarity structure the teacher learned, and that structure is a far richer training
signal per example.

## E.2 Response-based distillation — soft targets and temperature

Sharpened distributions carry little of that structure, so both teacher and student
logits $z$ are softened with a **temperature** $T$ before comparing:

$$p_i(T) = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}} \qquad
\mathcal{L} = \alpha \, T^2 \, \text{KL}\big(p^{teacher}(T) \,\|\, p^{student}(T)\big)
\; + \; (1-\alpha)\,\text{CE}(y, p^{student})$$

$T \in [2, 5]$ typically; the $T^2$ factor keeps gradient magnitudes comparable. The
same recipe drives modern **LLM distillation** — small chat models trained on the
token distributions (or just the sampled outputs — "sequence-level distillation") of
frontier models.

In [17]:
# Temperature in action: what softening reveals.
logits = np.array([9.0, 5.5, 4.8, 0.5])         # a confident teacher: cat, dog, fox, car
classes = ["cat", "dog", "fox", "car"]

def softmax_T(z, T):
    z = z / T
    e = np.exp(z - z.max())
    return e / e.sum()

print("class     " + "".join(f"{c:>8}" for c in classes))
for T in [1, 2, 5]:
    p = softmax_T(logits, T)
    print(f"T = {T:<4}  " + "".join(f"{x:8.3f}" for x in p))
print()
print("At T=1 the signal is ~'cat'. At T=5 the student also learns that dogs and foxes")
print("are cat-like while cars are not -- the 'dark knowledge' between the classes.")

class          cat     dog     fox     car
T = 1        0.957   0.029   0.014   0.000
T = 2        0.763   0.133   0.093   0.011
T = 5        0.474   0.235   0.205   0.087

At T=1 the signal is ~'cat'. At T=5 the student also learns that dogs and foxes
are cat-like while cars are not -- the 'dark knowledge' between the classes.


## E.3 Beyond logits — feature and relation distillation

- **Feature-based**: match intermediate representations — hidden states, attention maps
  — layer by layer (TinyBERT), usually through a learned projection when widths differ.
- **Relation-based**: match *relationships between* examples or positions rather than
  raw activations. The important instance for us:

**MiniLM — deep self-attention distillation** (Wang et al., 2020). Distill only the
**last transformer layer's self-attention**: the student matches the teacher's attention
distributions (the softmaxed $QK^\top$ from Part A!) *and* the value-value relation
matrix ($VV^\top$). Because only attention *patterns* are matched — not raw hidden
vectors — the student's depth and width are unconstrained. Result: 6 layers, 384 dims,
~33M parameters retaining ~99% of the teacher's accuracy on several benchmarks.

> **`all-MiniLM-L6-v2` — this pipeline's embedding model — is that distilled student**,
> subsequently contrastively fine-tuned (Part B.3 recipe) on 1B+ sentence pairs. The
> reason your laptop embeds thousands of chunks per second is distillation.

Other landmarks: **DistilBERT** (40% smaller, 60% faster, ~97% of BERT — logit KL +
cosine alignment of hidden states, student initialized from alternating teacher layers).

## E.4 Distillation for embedding/retrieval models specifically

1. **Cross-encoder → bi-encoder** — the standard trick behind strong open retrievers
   (the `ms-marco` sentence-transformers models): the cross-encoder (B.4, high quality,
   too slow to search with) scores (query, passage) pairs; the bi-encoder student
   trains to reproduce those scores (often via MarginMSE). You search at bi-encoder
   speed with much of the cross-encoder's judgment.
2. **Embedding-space imitation**: student trained so its vectors match the teacher's
   (MSE on embeddings) — also how multilingual models are made (align other languages'
   student vectors to the English teacher's space).
3. **Dimension reduction as mini-distillation**: append a learned projection (or PCA)
   to compress vectors; MRL (B.6) bakes the same idea into training. The demo below
   measures exactly what that trade-off looks like on our corpus.

In [18]:
# Poor-man's dimensionality distillation: PCA-compress our corpus embeddings,
# then check how well the compressed space preserves retrieval behaviour.
# (PCA can produce at most n_samples components; our demo corpus has 26 docs,
#  so we compress 384 -> 16 dims -- a 24x smaller index.)
from sklearn.decomposition import PCA

full = doc_dense                                   # (26, 384) from Part C, normalized
n_dims = 16
pca = PCA(n_components=n_dims, random_state=0).fit(full)

def compress(vectors):
    reduced = pca.transform(np.atleast_2d(vectors))
    return reduced / np.linalg.norm(reduced, axis=1, keepdims=True)

small = compress(full)

test_queries = [
    "What is the attention mechanism?",
    "hard negative mining techniques",
    "Which model has the largest context window?",
    "How does RAG reduce hallucination?",
]
overlaps = []
for q in test_queries:
    qv = st_model.encode([q], normalize_embeddings=True)[0]
    top_full = set(np.argsort(-(full @ qv))[:3])
    top_small = set(np.argsort(-(small @ compress(qv)[0]))[:3])
    overlap = len(top_full & top_small) / 3
    overlaps.append(overlap)
    print(f"top-3 overlap {overlap:.0%}  |  {q}")

print()
print(f"384 -> {n_dims} dims = {384 // n_dims}x smaller index, mean top-3 agreement "
      f"{np.mean(overlaps):.0%} on this corpus.")
print("Trained-in compression (MRL / learned projections) does better than PCA --")
print("this is the size/quality dial that distillation techniques let you turn.")

top-3 overlap 100%  |  What is the attention mechanism?


top-3 overlap 67%  |  hard negative mining techniques
top-3 overlap 100%  |  Which model has the largest context window?
top-3 overlap 67%  |  How does RAG reduce hallucination?

384 -> 16 dims = 24x smaller index, mean top-3 agreement 83% on this corpus.
Trained-in compression (MRL / learned projections) does better than PCA --
this is the size/quality dial that distillation techniques let you turn.


## E.5 Distillation vs the other compression levers

| Lever | What shrinks | Retraining? | Typical cost |
|---|---|---|---|
| **Distillation** | parameter count (new smaller model) | yes — full training run | large one-off, best quality/size |
| **Quantization** | bytes per weight (fp16→int8/int4) | no (PTQ) or light (QAT) | near-free, small quality dip |
| **Pruning** | weights/heads/layers removed | usually + recovery fine-tune | medium |
| **MRL / dim. reduction** | embedding vector size (index RAM) | trained-in or post-hoc | cheap |

They compose: a distilled student, quantized, with truncated embeddings, is the normal
production endpoint. Rule of thumb: **quantize first** (nearly free), **distill when
you own the training loop or a smaller distilled checkpoint already exists** (it
usually does — MiniLM being the proof in this very pipeline).

---
# Where each concept lives in this project

| Concept | Where you already used it |
|---|---|
| Transformer encoder + WordPiece + mean pooling | inside every `model.encode()` call |
| Scaled dot-product attention | the content of `attention_is_all_you_need.pdf` — and Part A's NumPy demo |
| Contrastive training + hard negatives | how `all-MiniLM-L6-v2` was fine-tuned; sample content in `embedding_models_report.pdf` |
| cosine ↔ L2 equivalence | ChromaDB cosine space (notebook 02) vs FAISS `IndexFlatL2` (`src/vector_store.py`) |
| Recursive chunking 1000/200 | `split_documents()` and `EmbeddingPipeline.chunk_documents()` |
| Distillation | the reason the embedding model is 33M params and fast on CPU |

# Reading list (in dependency order)

1. Vaswani et al. 2017 — *Attention Is All You Need*
2. Devlin et al. 2018 — *BERT*
3. Reimers & Gurevych 2019 — *Sentence-BERT*
4. Hinton et al. 2015 — *Distilling the Knowledge in a Neural Network*
5. Sanh et al. 2019 — *DistilBERT*; Wang et al. 2020 — *MiniLM*
6. Karpukhin et al. 2020 — *Dense Passage Retrieval*
7. Khattab & Zaharia 2020 — *ColBERT*; Formal et al. 2021 — *SPLADE*
8. Kusupati et al. 2022 — *Matryoshka Representation Learning*
9. Anthropic 2024 — *Contextual Retrieval* (blog); Sarthi et al. 2024 — *RAPTOR*